# Phase 7 — Stage C: hard top-1 fine-tuning

Stage C initializes from the completed Stage B checkpoint, locks routing to one expert, and fine-tunes with the adaptive entropy gate. It writes atomic resumable checkpoints, per-task loss/PSNR/SSIM matrices, expert-load and entropy statistics, coherence measurements, and trainer status to Google Drive. Rerunning the training cell resumes from this Stage C output directory. DIV2K is stored in Drive so it remains available after a Colab runtime ends. No prediction/intermediate images are written.

DIV2K downloads are optional and require accepting its academic-research terms. Other clean-image directories may be mounted from Drive after obtaining them under their applicable licenses. Real paired rows are supported by an optional CSV with `image_path,low_quality_path,task` columns.

In [ ]:
from pathlib import Path
import subprocess
import sys

from google.colab import drive
drive.mount('/content/drive')
REPO = Path('/content/SparseDict')
DATA_ROOT = Path('/content/drive/MyDrive/SparseDict/datasets/DIV2K')
RUN_DIR = Path('/content/drive/MyDrive/SparseDict/phase7/stage_c')
STAGE_B = Path('/content/drive/MyDrive/SparseDict/phase7/stage_b/latest.pt')
RUN_DIR.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', 'master', 'https://github.com/kalpeshjp89/SparseDict.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'master'], check=True)
sys.path.insert(0, str(REPO))
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab before training.')

In [ ]:
# Set True to download/extract all 800 DIV2K training-HR images (~3.28 GiB archive).
DOWNLOAD_DIV2K = False
if DOWNLOAD_DIV2K:
    subprocess.run([
        sys.executable, str(REPO / 'scripts' / 'download_div2k.py'),
        '--output-dir', str(DATA_ROOT), '--image-count', '800', '--accept-license'
    ], check=True)

In [ ]:
IMAGE_DIRS = [DATA_ROOT / 'DIV2K_train_HR']
# Append clean-image directories on Drive when available:
# IMAGE_DIRS += [Path('/content/drive/MyDrive/datasets/Flickr2K/HR')]
# IMAGE_DIRS += [Path('/content/drive/MyDrive/datasets/CBSD68'), Path('/content/drive/MyDrive/datasets/SOTS/clear')]
# IMAGE_DIRS += [Path('/content/drive/MyDrive/datasets/OutdoorRain/clean')]
PAIRED_MANIFESTS = []
# Example: PAIRED_MANIFESTS.append(Path('/content/drive/MyDrive/datasets/paired_train.csv'))
if not all(path.is_dir() for path in IMAGE_DIRS):
    raise FileNotFoundError('Download DIV2K or configure IMAGE_DIRS first.')

TRAIN_MANIFEST = RUN_DIR / 'train.csv'
VALIDATION_MANIFEST = RUN_DIR / 'validation.csv'
TASKS = 'sr2,sr3,sr4,denoise15,denoise25,denoise50,derain,desnow,dehaze,lowlight,jpeg'
manifest_command = [
    sys.executable, str(REPO / 'scripts' / 'make_universal_manifest.py'),
    '--train-output', str(TRAIN_MANIFEST), '--validation-output', str(VALIDATION_MANIFEST),
    '--validation-fraction', '0.1', '--seed', '2027', '--tasks', TASKS
]
for directory in IMAGE_DIRS:
    manifest_command.extend(['--image-dir', str(directory)])
for paired_manifest in PAIRED_MANIFESTS:
    manifest_command.extend(['--paired-manifest', str(paired_manifest)])
subprocess.run(manifest_command, cwd=REPO, check=True)
if not STAGE_B.is_file():
    raise FileNotFoundError(f'Run Stage B first: {STAGE_B}')

In [ ]:
command = [
    sys.executable, str(REPO / 'experiments' / 'train_universal.py'),
    '--phase', 'fine_tune', '--train-manifest', str(TRAIN_MANIFEST),
    '--validation-manifest', str(VALIDATION_MANIFEST), '--init-checkpoint', str(STAGE_B),
    '--output-dir', str(RUN_DIR), '--epochs', '20', '--warmup-epochs', '0',
    '--steps-per-epoch', '100', '--patch-size', '128',
    '--validation-samples-per-task', '8',
    '--checkpoint-every-steps', '100', '--device', 'cuda'
]
print(' '.join(command))
subprocess.run(command, cwd=REPO, check=True)
print('Metrics and status:', RUN_DIR / 'metrics.jsonl', RUN_DIR / 'metrics_by_task.csv', RUN_DIR / 'trainer_status.json')